<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/10_Semana_10_Sesion_02_Aplicacion_Regresion_Multiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Llegamos a la sesión donde la teoría multivariada se convierte en una herramienta analítica real.

Aquí es donde los estudiantes aprenderán a identificar qué variables realmente aportan valor y cuáles solo generan "ruido" o redundancia matemática.

### 📘 Estructura del Notebook: Semana 10 - Sesión 2

**Título del Notebook:** `Semana_10_Sesion_02_Aplicacion_Regresion_Multiple.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**

# SEMANA 10: Regresión Lineal Múltiple
**Asignatura:** Estadística Aplicada con Python y R
**Sesión 2:** Aplicación, Selección de Variables y Multicolinealidad.

## Fase 1: Actividad "Estudia y Aprende" (Aplicación)
Antes de programar, interactúa con tu IA de preferencia para comprender cómo interpretar los reportes estadísticos avanzados y cómo las métricas de penalización nos ayudan a elegir el mejor modelo.

**PROMPT 2 - APLICACIÓN:**
> Actúa como tutor experto en regresión múltiple con Python y R.
> 1. Muéstrame cómo ajustar un modelo múltiple.
> 2. Cómo interpretar coeficientes parciales.
> 3. Cómo detectar multicolinealidad.
> 4. Cómo calcular e interpretar R-cuadrado ajustado.
> 5. Cómo aplicar selección de variables con AIC.
> 6. Interpreta resultados como lo haría un ingeniero.
> No solo muestres código; explica qué significa cada resultado.

#### **Celda 2: Markdown (Fase 2: Demostración Docente - Contexto)**

## Fase 2: Demostración Docente en Python
Ajustaremos un modelo de regresión múltiple utilizando `statsmodels.OLS`. Nuestro enfoque estará en responder a la pregunta clave: **¿Este modelo explica realmente el fenómeno o solo sobreajusta los datos?**

Analizaremos:
1. **Significancia individual:** El Valor-p de cada coeficiente parcial.
2. **Significancia global:** El F-test (¿El modelo en su conjunto sirve para algo?).
3. **$R^2$ ajustado:** La métrica real del éxito multivariado.
4. **Factor de Inflación de la Varianza (VIF):** Nuestra herramienta principal para detectar Multicolinealidad.

#### **Celda 3: Código Python (Ajuste del Modelo Múltiple y Summary)**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(123)

# 1. GENERAMOS DATOS (Contexto: Resistencia del concreto en Ingeniería Civil)
# Variables explicativas:
cemento_X1 = np.random.uniform(250, 400, 100) # kg/m3
agua_X2 = np.random.uniform(150, 220, 100)    # litros/m3
# Introducimos una variable redundante (Multicolinealidad intencional)
ceniza_volante_X3 = cemento_X1 * 0.15 + np.random.normal(0, 5, 100)

# Variable respuesta: Resistencia (MPa)
resistencia_Y = 10 + 0.08 * cemento_X1 - 0.05 * agua_X2 + np.random.normal(0, 3, 100)

df_concreto = pd.DataFrame({
    'Cemento': cemento_X1,
    'Agua': agua_X2,
    'Ceniza': ceniza_volante_X3,
    'Resistencia': resistencia_Y
})

# 2. AJUSTE DEL MODELO MÚLTIPLE
X_multi = df_concreto[['Cemento', 'Agua', 'Ceniza']]
X_multi_const = sm.add_constant(X_multi)
Y = df_concreto['Resistencia']

modelo_multi = sm.OLS(Y, X_multi_const).fit()
print(modelo_multi.summary())

# Interpretación Ingenieril:
# - F-statistic (Prob): Es muy pequeño, el modelo GLOBALMENTE es útil.
# - R-squared Adj: Fíjate en el "Adj. R-squared", penaliza si agregamos variables inútiles.
# - P>|t| individuales: El 'Cemento' y el 'Agua' son significativos (< 0.05).
#   Pero la 'Ceniza' tiene un valor-p alto. El modelo nos dice que no aporta información nueva.

#### **Celda 4: Código Python (Cálculo del VIF para Multicolinealidad)**

In [ ]:
# 3. DETECCIÓN DE MULTICOLINEALIDAD (VIF)
# Calculamos el Factor de Inflación de Varianza (VIF) para cada variable explicativa.
# Regla de oro en ingeniería: VIF > 5 o 10 indica multicolinealidad severa.

vif_data = pd.DataFrame()
vif_data["Variable"] = X_multi.columns

# Calculamos el VIF usando la función de statsmodels
vif_data["VIF"] = [variance_inflation_factor(X_multi.values, i) for i in range(len(X_multi.columns))]

print("\n--- FACTOR DE INFLACIÓN DE LA VARIANZA (VIF) ---")
print(vif_data)

# Interpretación:
# Observa cómo el Cemento y la Ceniza tienen valores de VIF extremadamente altos.
# Esto confirma lo que sospechábamos: están aportando la misma información al modelo.
# Acción: Para mejorar el modelo, un ingeniero eliminaría la 'Ceniza' y volvería a ajustar.

#### **Celda 5: Markdown (Transición a R, RMarkdown y stepAIC)**

```markdown
## Trabajo Autónomo: Transición a R, stepAIC y RPubs

R brilla en la regresión múltiple gracias a paquetes como `MASS` para la selección automática de variables y `car` para diagnósticos como el VIF.

**Paso 1: Validación en Colab (Entorno R)**
1. Crea un nuevo Notebook en Colab, asegúrate de cambiar el entorno a **R**.
2. Ejecuta el siguiente código. Observa cómo `stepAIC()` evalúa automáticamente combinaciones de variables y se queda con el mejor modelo penalizado.

```R
# Instalar paquetes si no están disponibles
# install.packages(c("MASS", "car"))
library(MASS)
library(car)

# Recrear los datos en R
set.seed(123)
cemento <- runif(100, 250, 400)
agua <- runif(100, 150, 220)
ceniza <- cemento * 0.15 + rnorm(100, 0, 5)
resistencia <- 10 + 0.08 * cemento - 0.05 * agua + rnorm(100, 0, 3)

datos_r <- data.frame(Cemento=cemento, Agua=agua, Ceniza=ceniza, Resistencia=resistencia)

# 1. Ajustar el modelo completo con todas las variables
modelo_completo <- lm(Resistencia ~ Cemento + Agua + Ceniza, data=datos_r)
print("--- VIF DEL MODELO COMPLETO ---")
print(vif(modelo_completo)) # Si es mayor a 5, hay multicolinealidad

# 2. Selección paso a paso (Stepwise) usando AIC
print("--- SELECCIÓN DE VARIABLES CON stepAIC ---")
modelo_optimo <- stepAIC(modelo_completo, direction="both", trace=FALSE)
summary(modelo_optimo)

# Interpretación: R elimina automáticamente la variable 'Ceniza' porque el AIC
# (Criterio de Información de Akaike) determina que mantenerla penaliza más de lo que ayuda.

```

**Paso 2: Documentación en Posit Cloud y Despliegue en RPubs**

1. Abre tu espacio en **Posit Cloud** y crea un documento **RMarkdown**.
2. Transfiere el código de R y ejecútalo.
3. Redacta tu análisis: *Explica en un párrafo por qué confiarías más en el $R^2$ ajustado que en el $R^2$ normal al evaluar la calidad del modelo final generado por `stepAIC()`. ¿Qué nos dice el VIF sobre la redundancia de datos?*
4. Compila a HTML y publica en **RPubs**.

```

#### **Celda 6: Markdown (CIERRE DEL TEMA Y HOJA EVALUABLE)**
```markdown
---
## CIERRE DEL TEMA: Generación de Resumen Guía

Hemos terminado la regresión lineal múltiple. Es el momento de sintetizar tus conocimientos usando tu IA para preparar la evidencia escrita.

**PROMPT DE CIERRE GLOBAL:**
> Genera un resumen estructurado para escribir a mano en UNA sola hoja.
> Formato obligatorio:
> A) Idea central (1-2 líneas).
> B) 6-10 viñetas organizadas lógicamente.
> C) 3 relaciones clave (por qué/cómo).
> D) 1 ejemplo aplicado a ingeniería.
> E) 3 preguntas de autoevaluación con respuesta.
> F) Cierre: "Hoy aprendí que ..."

⚠️ **RECUERDA (EVIDENCIA EVALUABLE):**
En el reverso de tu hoja física titulada *"Regresión Lineal Múltiple en Ingeniería"*, escribe a mano:
* La idea central del modelado con múltiples variables.
* El modelo matemático general explicado.
* La interpretación correcta de los coeficientes parciales (controlando por las demás variables).
* El concepto de Multicolinealidad y el uso del VIF.
* La diferencia crítica entre $R^2$ y $R^2$ ajustado.
* El uso de AIC/BIC para selección de variables.
* Un ejemplo aplicado.
* Las 3 preguntas con sus respuestas y tu reflexión final.

```

---

¡Semana 10 completada! Los estudiantes ahora tienen el criterio estadístico para no dejarse engañar por modelos sobreajustados.